In [ ]:
%matplotlib widget

In [ ]:
from glob import glob
import numpy as np
from scipy import stats
import pandas as pd
import flammkuchen as fl
from split_dataset import SplitDataset
from bouter import Experiment
from motions.utilities import stim_vel_dir_dataframe, quantize_directions
from scipy.interpolate import interp1d 
from scipy.signal import convolve2d
import colorspacious
import napari
import matplotlib.pyplot as plt

from fimpylab.core.twop_experiment import TwoPExperiment

from pathlib import Path

In [ ]:
master_ablated = Path(r"Z:\Hagar\e0075\ablation\ntr")
master_control = Path(r"Z:\Hagar\e0075\ablation\control")

fish_list_control_pre = list(master_control.glob("*_pre*"))
fish_list_ablated_pre = list(master_ablated.glob("*_pre*"))

fish_list_control_post = list(master_control.glob("*_post*"))
fish_list_ablated_post = list(master_ablated.glob("*_post*"))


In [ ]:
num_ntr = len(fish_list_ablated_pre)
num_control = len(fish_list_control_pre)

print(num_ntr)
print(num_control)

In [ ]:
## single fish plot

In [ ]:
fish_list_ablated_pre

In [ ]:
fish_list_ablated_post

In [ ]:
fish_list_control_pre

In [ ]:
fish_list_control_post

In [ ]:
count = 0

for fish in fish_list_ablated_pre:
    suite2p_path = fish / 'suite2p'
    path_list = list(suite2p_path.glob("*00*"))
    
    num_planes = len(path_list)
    
    for path in path_list:
        sens_regs = fl.load(path / 'sensory_regressors_cells.h5')['regressors_values']
        
        #apply mask
        ipn_coords = fl.load(path / 'ipn_coords.h5')['coords_to_keep']
        sens_regs = sens_regs[:, ipn_coords]
        
        if count == 0:
            all_reg_values_ntr_pre = sens_regs
            count = 1
        else:
            all_reg_values_ntr_pre = np.append(all_reg_values_ntr_pre, sens_regs)

In [ ]:
count = 0

for fish in fish_list_control_pre:
    suite2p_path = fish / 'suite2p'
    path_list = list(suite2p_path.glob("*00*"))
    
    num_planes = len(path_list)
    
    for path in path_list:
        sens_regs = fl.load(path / 'sensory_regressors_cells.h5')['regressors_values']
        
        #apply mask
        ipn_coords = fl.load(path / 'ipn_coords.h5')['coords_to_keep']
        sens_regs = sens_regs[:, ipn_coords]
        
        if count == 0:
            all_reg_values_control_pre = sens_regs
            count = 1
        else:
            all_reg_values_control_pre = np.append(all_reg_values_control_pre, sens_regs)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(8,5), sharex=True, sharey=True)

ax[0].set_title('NTR (n=8)')
ax[0].hist(all_reg_values_ntr_pre, bins=100, density=True, alpha=0.5)
#ax[0].set_xlim(-0.5, 0.5)
ax[0].spines['right'].set_visible(False)
ax[0].spines['top'].set_visible(False)

ax[0].set_xlabel('Correlation with all regressors')
ax[1].set_xlabel('Correlation with all regressors')
ax[0].set_ylabel('Density')

ax[1].set_title('Control (n=8)')
ax[1].hist(all_reg_values_control_pre, bins=100, density=True, alpha=0.5)
ax[1].spines['right'].set_visible(False)
ax[1].spines['top'].set_visible(False)

In [ ]:
count = 0

for fish in fish_list_ablated_post:
    suite2p_path = fish / 'suite2p'
    path_list = list(suite2p_path.glob("*00*"))
    
    num_planes = len(path_list)
    for path in path_list:
        sens_regs = fl.load(path / 'sensory_regressors_cells.h5')['regressors_values']
        
        #apply mask
        ipn_coords = fl.load(path / 'ipn_coords.h5')['coords_to_keep']
        sens_regs = sens_regs[:, ipn_coords]
        
        if count == 0:
            all_reg_values_ntr_post = sens_regs
            count = 1
        else:
            all_reg_values_ntr_post = np.append(all_reg_values_ntr_post, sens_regs)

In [ ]:
count = 0

for fish in fish_list_control_post:
    suite2p_path = fish / 'suite2p'
    path_list = list(suite2p_path.glob("*00*"))
    
    num_planes = len(path_list)
    for path in path_list:
        sens_regs = fl.load(path / 'sensory_regressors_cells.h5')['regressors_values']
        
        #apply mask
        ipn_coords = fl.load(path / 'ipn_coords.h5')['coords_to_keep']
        sens_regs = sens_regs[:, ipn_coords]
        
        if count == 0:
            all_reg_values_control_post = sens_regs
            count = 1
        else:
            all_reg_values_control_post = np.append(all_reg_values_control_post, sens_regs)

In [ ]:
ax[0].hist(all_reg_values_ntr_post, bins=100, density=True, alpha=0.5)
ax[0].set_xlim(-0.5, 0.5)
ax[0].spines['right'].set_visible(False)
ax[0].spines['top'].set_visible(False)


ax[1].hist(all_reg_values_control_post, bins=100, density=True, alpha=0.5)
ax[1].set_xlim(-0.5, 0.5)
ax[1].spines['right'].set_visible(False)
ax[1].spines['top'].set_visible(False)

In [ ]:
file_name = "Regressors corr distribution density n=" + str(num_ntr) + ".jpg"
fig.savefig(master_ablated / file_name, dpi=300)

file_name = "Regressors corr distribution density n=" + str(num_ntr) + ".pdf"
fig.savefig(master_ablated / file_name, dpi=300)

In [ ]:
# Calculate the F-statistic
f = np.var(all_reg_values_ntr_pre, ddof=1) / np.var(all_reg_values_ntr_post, ddof=1)

# Calculate degrees of freedom
dfn = len(all_reg_values_ntr_pre) - 1  # degrees of freedom numerator
dfd = len(all_reg_values_ntr_post) - 1  # degrees of freedom denominator

# Perform the F-test
p_value = stats.f.cdf(f, dfn, dfd)
p_value = 2 * min(p_value, 1 - p_value)  # two-tailed test

print(f"F-statistic: {f}")
print(f"p-value: {p_value}")

In [ ]:
# Calculate the F-statistic
f = np.var(all_reg_values_control_pre, ddof=1) / np.var(all_reg_values_control_post, ddof=1)

# Calculate degrees of freedom
dfn = len(all_reg_values_ntr_pre) - 1  # degrees of freedom numerator
dfd = len(all_reg_values_ntr_post) - 1  # degrees of freedom denominator

# Perform the F-test
p_value = stats.f.cdf(f, dfn, dfd)
p_value = 2 * min(p_value, 1 - p_value)  # two-tailed test

print(f"F-statistic: {f}")
print(f"p-value: {p_value}")